In [1]:
import torch
import torch.nn as nn
import torchvision.datasets as CIFAR10
import torchvision
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms

In [2]:
# dataset and dataloader

from logging import root


transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10.CIFAR10(root='./data', train=True, download=True, transform=transforms)
testset = CIFAR10.CIFAR10(root='./data', train=False, download=True, transform=transforms)

In [3]:
dataloader_train = DataLoader(trainset, batch_size=64, shuffle=True)
dataloader_test = DataLoader(testset, batch_size=64, shuffle=False)

In [ ]:
# Building the CNN model

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), #32x32x3 input, 32 filters, 3x3 kernel, padding=1 to maintain spatial dimensions
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), #stride=2 means that the window will move 2 pixels at a time, effectively reducing the spatial dimensions by half in each dimension 

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), 
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),  
            nn.MaxPool2d(kernel_size=2, stride=2)       
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [7]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    train_loss = 0.0

    for images, labels in dataloader_train:
        optimizer.zero_grad() # Zero the parameter gradients
        output = model(images) # Forward pass
        loss  = criterion(output, labels) # Compute loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights

        running_loss += loss.item()
        train_loss += loss.item()
    epoch_loss = running_loss / len(dataloader_train)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 1.3537
Epoch 2/10, Loss: 0.9334
Epoch 3/10, Loss: 0.7472
Epoch 4/10, Loss: 0.6216
Epoch 5/10, Loss: 0.5183
Epoch 6/10, Loss: 0.4205
Epoch 7/10, Loss: 0.3370
Epoch 8/10, Loss: 0.2639
Epoch 9/10, Loss: 0.2064
Epoch 10/10, Loss: 0.1593


In [9]:
model.eval()

val_loss = 0.0

with torch.no_grad():
    for images, labels in dataloader_test:
        output = model(images)
        loss = criterion(output, labels)
        val_loss += loss.item()
val_loss /= len(dataloader_test)
print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

Epoch 10/10, Train Loss: 124.5416, Val Loss: 1.0491


In [10]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        output = model(images)
        _, predicted = torch.max(output.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {(correct / total) * 100:.2f}%")

print(correct)
print(total)


Accuracy: 75.44%
7544
10000
